# Evaluate-all - compare every trained model

**Token models:**
* `token_model_baseline` - custom char-level tokenizer, baseline TokenModel
* `token_model_standart` - same architecture as baseline, with mixed-precision / cosine LR / label smoothing
* `token_model_bpe_rope` - byte-level BPE tokenizer + Rotary Position Embeddings
* `token_model_codegen`- pretrained Salesforce/codegen-350M-mono, fine-tuned

**Line models:**
* `line_model_baseline`- custom char-level + encoder-decoder
* `line_model_standart`- same, with modern training tricks
* `line_model_bpe_rope`- BPE + RoPE encoder-decoder
* `line_model_T5_small`- fine-tuned `Salesforce/codet5-small`
* `line_model_MLC_T5_small`- same model, multi-line context dataset
* `line_model_MLC_T5p_220m_py` - fine-tuned `Salesforce/codet5p-220m-py` with multi-line context

Metrics: syntax validity, bracket balance, basic-construction handling, first-token match. Exact-match and BLEU are for reference, they're not valid for autocomplete.

## Setup

In [1]:
import os, glob, random, torch, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from modules.evaluation import (
    evaluate_token_model,
    evaluate_line_model,
    compare_models,
)

#  adjust to your environment 
WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
# WORKDIR = '/home/ubuntu/code_autocomplete'

DATA_DIR  = f'{WORKDIR}/Clean_Dataset'
CKPT_ROOT = f'{WORKDIR}/checkpoints'
EVAL_ROOT = f'{WORKDIR}/results'
TOKENIZERS = f'{WORKDIR}/experiments'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def load_eval_texts(data_dir, n=200, seed=42):
    paths = sorted(glob.glob(os.path.join(data_dir, '**/*.py'), recursive=True))
    random.Random(seed).shuffle(paths)
    return [Path(p).read_text(errors='replace') for p in paths[:n]]

eval_texts = load_eval_texts(DATA_DIR, n=200)
print(f'Loaded {len(eval_texts)} eval texts')

# results accumulators
token_results = {}
line_results  = {}

def latest_ckpt(model_name):
    """Return the path to the lowest-loss checkpoint for a given model name, or None if missing."""
    paths = sorted(glob.glob(f'{CKPT_ROOT}/{model_name}/{model_name}_*.pt'))
    return paths[0] if paths else None

def report(name, ok, msg=''):
    status = '✓' if ok else '✗'
    print(f'  {status} {name}  {msg}')

Device: cuda
Loaded 200 eval texts


# Token models

## 1. Token - baseline  (custom char tokenizer + baseline TokenModel)

In [2]:
MODEL_NAME = 'token_model_baseline'
try:
    from modules.tokenizers.base_tokenizer import CodeTokenizer
    from modules.models.T_baseline_model import TokenModel, ModelCfg

    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'

    tokenizer = CodeTokenizer.load(f'{TOKENIZERS}/tokenizer.json')
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ck.get('cfg') or ModelCfg(
        vocab=tokenizer.vocab, d_model=256, n_heads=8, n_layers=4, d_ff=1024, max_len=160,
    )
    model = TokenModel(cfg).to(device)
    model.load_state_dict(ck['model_state'])

    res = evaluate_token_model(
        model=model, tokenizer=tokenizer, eval_texts=eval_texts, device=device,
        ctx=128, n_samples=2000, is_hf=False,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Token - Baseline',
    )
    token_results['baseline'] = res
    report(MODEL_NAME, True, f'top-1={res["top1_acc"]:.3f}  ppl={res["perplexity"]:.1f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

[Eval Token] tokenising 200 texts ...
[Eval Token] 2000 samples, ctx=128
[Eval Token] running 20 basic-construction prompts ...
[Eval Token] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_baseline\eval_token_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_baseline\eval_token_dashboard.png
  ✓ token_model_baseline  top-1=0.757  ppl=3.8


## 2. Token - standart  (mixed precision + warmup-cosine + label smoothing)

In [3]:
MODEL_NAME = 'token_model_standart'
try:
    from modules.tokenizers.base_tokenizer import CodeTokenizer
    from modules.models.T_baseline_model import TokenModel, ModelCfg

    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'

    tokenizer = CodeTokenizer.load(f'{TOKENIZERS}/tokenizer.json')
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ck.get('cfg') or ModelCfg(
        vocab=tokenizer.vocab, d_model=256, n_heads=8, n_layers=4, d_ff=1024, max_len=160,
    )
    model = TokenModel(cfg).to(device)
    model.load_state_dict(ck['model_state'])

    res = evaluate_token_model(
        model=model, tokenizer=tokenizer, eval_texts=eval_texts, device=device,
        ctx=128, n_samples=2000, is_hf=False,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Token - Standart',
    )
    token_results['standart'] = res
    report(MODEL_NAME, True, f'top-1={res["top1_acc"]:.3f}  ppl={res["perplexity"]:.1f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

[Eval Token] tokenising 200 texts ...
[Eval Token] 2000 samples, ctx=128
[Eval Token] running 20 basic-construction prompts ...
[Eval Token] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_standart\eval_token_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_standart\eval_token_dashboard.png
  ✓ token_model_standart  top-1=0.756  ppl=3.7


## 3. Token - BPE + RoPE

In [4]:
MODEL_NAME = 'token_model_bpe_rope'
try:
    from modules.tokenizers.BPE_tokenizer import BPECodeTokenizer
    from modules.models.T_rope_model import TokenModel, ModelCfg

    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'

    tokenizer = BPECodeTokenizer.load(f'{TOKENIZERS}/tokenizer_bpe.json')
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ck.get('cfg') or ModelCfg(
        vocab=tokenizer.vocab, d_model=256, n_heads=8, n_layers=4, d_ff=1024, max_len=160,
    )
    model = TokenModel(cfg).to(device)
    model.load_state_dict(ck['model_state'])

    res = evaluate_token_model(
        model=model, tokenizer=tokenizer, eval_texts=eval_texts, device=device,
        ctx=128, n_samples=2000, is_hf=False,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Token - BPE + RoPE',
    )
    token_results['bpe_rope'] = res
    report(MODEL_NAME, True, f'top-1={res["top1_acc"]:.3f}  ppl={res["perplexity"]:.1f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

[Eval Token] tokenising 200 texts ...
[Eval Token] 2000 samples, ctx=128
[Eval Token] running 20 basic-construction prompts ...
[Eval Token] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_bpe_rope\eval_token_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_bpe_rope\eval_token_dashboard.png
  ✓ token_model_bpe_rope  top-1=0.485  ppl=41.5


## 4. Token - CodeGen-350M-mono (pretrained, fine-tuned)

In [5]:
MODEL_NAME = 'token_model_codegen'
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    HF_MODEL = 'Salesforce/codegen-350M-mono'
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)
    if hf_tok.pad_token is None:
        hf_tok.pad_token = hf_tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(HF_MODEL).to(device)

    
    print("  NOT FINE-TUNED:")
    res = evaluate_token_model(
        model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
        ctx=256, n_samples=1000, is_hf=True,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='NO-LOAD Token - CodeGen-350M',
        no_load=True,
    )
    token_results['NO-LOAD-codegen_350m'] = res
    report("NO-LOAD" + MODEL_NAME, True, f'top-1={res["top1_acc"]:.3f}  ppl={res["perplexity"]:.1f}')


    ckpt_path = latest_ckpt(MODEL_NAME)
    if ckpt_path:
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ck['model_state'])
        print("  FINE-TUNED:")
        print(f'  LOADED fine-tuned weights from {ckpt_path}')
        res = evaluate_token_model(
            model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
            ctx=256, n_samples=1000, is_hf=True,
            out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Token - CodeGen-350M',
        )
        token_results['codegen_350m'] = res
        report(MODEL_NAME, True, f'top-1={res["top1_acc"]:.3f}  ppl={res["perplexity"]:.1f}')
    else:
        print(f'  no fine-tuned checkpoint')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

Token indices sequence length is longer than the specified maximum sequence length for this model (2452 > 2048). Running this sequence through the model will result in indexing errors


  NOT FINE-TUNED:
[Eval Token] tokenising 200 texts ...
[Eval Token] 1000 samples, ctx=256


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[Eval Token] running 20 basic-construction prompts ...
[Eval Token] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_codegen\NO-LOAD eval_token_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/token_model_codegen\NO-LOAD eval_token_dashboard.png
  ✓ NO-LOADtoken_model_codegen  top-1=0.773  ppl=3.6
  no fine-tuned checkpoint


## Token models - comparison dashboard

In [6]:
if len(token_results) >= 2:
    compare_models(
        results = token_results,
        out_path = f'{EVAL_ROOT}/comparisons/compare_token_all.png',
        kind = 'token',
    )
    print(f'Compared {len(token_results)} token models: {list(token_results.keys())}')
else:
    print(f'Only {len(token_results)} token model(s) evaluated - need >=2 to compare')

[Compare] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/comparisons/compare_token_all.png
Compared 4 token models: ['baseline', 'standart', 'bpe_rope', 'NO-LOAD-codegen_350m']


# Line models

## 5. Line - baseline  (custom char + encoder-decoder)

In [7]:
MODEL_NAME = 'line_model_baseline'
try:
    from modules.tokenizers.base_tokenizer import CodeTokenizer
    from modules.models.L_baseline_model import LineModel, ModelCfg

    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'

    tokenizer = CodeTokenizer.load(f'{TOKENIZERS}/tokenizer.json')
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ck.get('cfg') or ModelCfg(
        vocab=tokenizer.vocab, d_model=256, n_heads=8, n_layers=4, d_ff=1024, max_len=160,
    )
    model = LineModel(cfg).to(device)
    model.load_state_dict(ck['model_state'])

    res = evaluate_line_model(
        model=model, tokenizer=tokenizer, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=False,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Line - Baseline',
    )
    line_results['baseline'] = res
    report(MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples
[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_baseline\eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_baseline\eval_line_dashboard.png
  ✓ line_model_baseline  syntax=0.088  brackets=0.948  construct=0.400  first_tok=0.026


## 6. Line - standart

In [8]:
MODEL_NAME = 'line_model_standart'
try:
    from modules.tokenizers.base_tokenizer import CodeTokenizer
    from modules.models.L_baseline_model import LineModel, ModelCfg

    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'

    tokenizer = CodeTokenizer.load(f'{TOKENIZERS}/tokenizer.json')
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ck.get('cfg') or ModelCfg(
        vocab=tokenizer.vocab, d_model=256, n_heads=8, n_layers=4, d_ff=1024, max_len=160,
    )
    model = LineModel(cfg).to(device)
    model.load_state_dict(ck['model_state'])

    res = evaluate_line_model(
        model=model, tokenizer=tokenizer, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=False,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Line - Standart',
    )
    line_results['standart'] = res
    report(MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples
[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_standart\eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_standart\eval_line_dashboard.png
  ✓ line_model_standart  syntax=0.064  brackets=0.886  construct=0.400  first_tok=0.032


## 7. Line - BPE + RoPE

In [9]:
MODEL_NAME = 'line_model_bpe_rope'
try:
    from modules.tokenizers.BPE_tokenizer import BPECodeTokenizer
    from modules.models.L_rope_model import LineModel, ModelCfg

    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'

    tokenizer = BPECodeTokenizer.load(f'{TOKENIZERS}/tokenizer_bpe.json')
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ck.get('cfg') or ModelCfg(
        vocab=tokenizer.vocab, d_model=256, n_heads=8, n_layers=4, d_ff=1024, max_len=160,
    )
    model = LineModel(cfg).to(device)
    model.load_state_dict(ck['model_state'])

    res = evaluate_line_model(
        model=model, tokenizer=tokenizer, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=False,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Line - BPE + RoPE',
    )
    line_results['bpe_rope'] = res
    report(MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples
[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_bpe_rope\eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_bpe_rope\eval_line_dashboard.png
  ✓ line_model_bpe_rope  syntax=0.070  brackets=0.828  construct=0.050  first_tok=0.000


## 8. Line - CodeT5-small (single-line context)

In [10]:
MODEL_NAME = 'line_model_T5_small'
try:
    from transformers import T5ForConditionalGeneration, AutoTokenizer

    HF_MODEL = 'Salesforce/codet5-small'
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)
    model  = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)

    print("  NOT FINE-TUNED:")
    res = evaluate_line_model(
        model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=True,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='NO-LOAD Line - CodeT5-small',
        no_load=True,
    )
    line_results['NO-LOAD-codet5_small'] = res
    report("NO-LOAD" + MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')

    ckpt_path = latest_ckpt(MODEL_NAME)
    if ckpt_path:
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ck['model_state'])
        print("  FINE-TUNED:")
        print(f'  loaded fine-tuned weights from {ckpt_path}')
        res = evaluate_line_model(
            model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
            n_samples=500, is_hf_seq2seq=True,
            out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Line - CodeT5-small',
        )
        line_results['codet5_small'] = res
        report(MODEL_NAME, True,
                f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
                f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')
    else:
        print(f'  no fine-tuned checkpoint - evaluating pretrained baseline')

except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

  NOT FINE-TUNED:
[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples
[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_T5_small\NO-LOAD eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_T5_small\NO-LOAD eval_line_dashboard.png
  ✓ NO-LOADline_model_T5_small  syntax=0.084  brackets=0.852  construct=0.800  first_tok=0.000
  FINE-TUNED:
  loaded fine-tuned weights from C:\Users\Roman\Documents\Projects\code_autocomplete/checkpoints/line_model_T5_small\line_model_T5_small_ep003_loss3.0416.pt
[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples
[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_T5_small\eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_T

## 9. Line - CodeT5-small + multi-line context

In [11]:
MODEL_NAME = 'line_model_MLC_T5_small'
try:
    from transformers import T5ForConditionalGeneration, AutoTokenizer

    HF_MODEL = 'Salesforce/codet5-small'
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)
    model  = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)


    print("  NOT FINE-TUNED:")
    res = evaluate_line_model(
        model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=True,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='NO-LOAD Line - CodeT5-small + MLC',
        no_load=True,
    )
    line_results['NO-LOAD-codet5_small_mlc'] = res
    report("NO-LOAD" + MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')



    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'
    print("  FINE-TUNED:")
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state'])

    res = evaluate_line_model(
        model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=True,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Line - CodeT5-small + MLC',
    )
    line_results['codet5_small_mlc'] = res
    report(MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

  NOT FINE-TUNED:
[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples


Token indices sequence length is longer than the specified maximum sequence length for this model (557 > 512). Running this sequence through the model will result in indexing errors


[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5_small\NO-LOAD eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5_small\NO-LOAD eval_line_dashboard.png
  ✓ NO-LOADline_model_MLC_T5_small  syntax=0.072  brackets=0.796  construct=0.800  first_tok=0.000
  FINE-TUNED:
[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples
[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5_small\eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5_small\eval_line_dashboard.png
  ✓ line_model_MLC_T5_small  syntax=0.082  brackets=0.860  construct=0.800  first_tok=0.016


## 10. Line - CodeT5+ 220m-py + multi-line context

In [12]:
MODEL_NAME = 'line_model_MLC_T5p_220m_py'
try:
    from transformers import T5ForConditionalGeneration, AutoTokenizer

    HF_MODEL = 'Salesforce/codet5p-220m-py'
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)
    model  = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)


    print("  NOT FINE-TUNED:")
    res = evaluate_line_model(
        model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=True,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='NO-LOAD Line - CodeT5+ 220m-py + MLC',
        no_load=True,
    )
    line_results['NO-LOAD-codet5p_220m_mlc'] = res
    report("NO-LOAD" + MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')


    ckpt_path = latest_ckpt(MODEL_NAME)
    assert ckpt_path, f'No checkpoint found in {CKPT_ROOT}/{MODEL_NAME}/'
    print("  FINE-TUNED:")
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state'])

    res = evaluate_line_model(
        model=model, tokenizer=hf_tok, eval_texts=eval_texts, device=device,
        n_samples=500, is_hf_seq2seq=True,
        out_dir=f'{EVAL_ROOT}/{MODEL_NAME}', title='Line - CodeT5+ 220m-py + MLC',
    )
    line_results['codet5p_220m_mlc'] = res
    report(MODEL_NAME, True,
           f'syntax={res["syntax_valid"]:.3f}  brackets={res["bracket_balance"]:.3f}  '
           f'construct={res.get("construction_acceptable", 0):.3f}  first_tok={res["first_token_match"]:.3f}')
except Exception as e:
    report(MODEL_NAME, False, f'skipped - {type(e).__name__}: {e}')

  NOT FINE-TUNED:
[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples
[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5p_220m_py\NO-LOAD eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5p_220m_py\NO-LOAD eval_line_dashboard.png
  ✓ NO-LOADline_model_MLC_T5p_220m_py  syntax=0.066  brackets=0.806  construct=0.650  first_tok=0.000
  FINE-TUNED:
[Eval Line] building samples from 200 texts ...
[Eval Line] 500 samples


Token indices sequence length is longer than the specified maximum sequence length for this model (619 > 512). Running this sequence through the model will result in indexing errors


[Eval Line] running 20 basic-construction prompts ...
[Eval Line] results -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5p_220m_py\eval_line_results.json
[Plot] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/line_model_MLC_T5p_220m_py\eval_line_dashboard.png
  ✓ line_model_MLC_T5p_220m_py  syntax=0.074  brackets=0.780  construct=0.400  first_tok=0.002


## Line models - comparison dashboard

In [13]:
if len(line_results) >= 2:
    compare_models(
        results = line_results,
        out_path = f'{EVAL_ROOT}/comparisons/compare_line_all.png',
        kind = 'line',
    )
    print(f'Compared {len(line_results)} line models: {list(line_results.keys())}')
else:
    print(f'Only {len(line_results)} line model(s) evaluated - need >=2 to compare')

[Compare] -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/comparisons/compare_line_all.png
Compared 9 line models: ['baseline', 'standart', 'bpe_rope', 'NO-LOAD-codet5_small', 'codet5_small', 'NO-LOAD-codet5_small_mlc', 'codet5_small_mlc', 'NO-LOAD-codet5p_220m_mlc', 'codet5p_220m_mlc']


## Summary table

In [16]:
def fmt_token_row(name, r):
    return (f'  {name:24s}  top1={r["top1_acc"]:.3f}  top5={r["top5_acc"]:.3f}  '
            f'construct={r.get("construction_acceptable", 0):.3f}  '
            f'syntax={r.get("construction_syntax", 0):.3f}  '
            f'ppl={r["perplexity"]:7.1f}  p50={r["latency_p50"]:6.1f}ms')

def fmt_line_row(name, r):
    return (f'  {name:24s}  syntax={r["syntax_valid"]:.3f}  brackets={r["bracket_balance"]:.3f}  '
            f'construct={r.get("construction_acceptable", 0):.3f}  '
            f'first_tok={r["first_token_match"]:.3f}  '
            f'p50={r["latency_p50"]:6.1f}ms')

# composite autocomplete score for ranking line models
def line_score(r):
    return (0.35 * r["syntax_valid"] + 0.25 * r["bracket_balance"]
            + 0.20 * r.get("construction_acceptable", 0) + 0.20 * r["first_token_match"])

print('TOKEN MODELS  (next-token prediction)  -- sorted by basic-construction acceptability')
if token_results:
    for name, r in sorted(token_results.items(),
                          key=lambda kv: -kv[1].get('construction_acceptable', 0)):
        print(fmt_token_row(name, r))
else:
    print('  (none evaluated)')

print()
print('LINE MODELS  (line completion)  -- sorted by composite autocomplete score')
if line_results:
    for name, r in sorted(line_results.items(), key=lambda kv: -line_score(kv[1])):
        print(fmt_line_row(name, r))
else:
    print('  (none evaluated)')

print()
print(f'All dashboards saved to {EVAL_ROOT}/')
print(f'  per-model:   {EVAL_ROOT}/<model_name>/eval_*_dashboard.png')
print(f'  comparisons: {EVAL_ROOT}/comparisons/compare_{{token,line}}_all.png')


TOKEN MODELS  (next-token prediction)  -- sorted by basic-construction acceptability
  NO-LOAD-codegen_350m      top1=0.773  top5=0.910  construct=0.900  syntax=0.150  ppl=    3.6  p50=  51.8ms
  baseline                  top1=0.757  top5=0.888  construct=0.050  syntax=0.050  ppl=    3.8  p50=   2.3ms
  standart                  top1=0.756  top5=0.889  construct=0.050  syntax=0.050  ppl=    3.7  p50=   2.5ms
  bpe_rope                  top1=0.485  top5=0.661  construct=0.050  syntax=0.050  ppl=   41.5  p50=   3.3ms

LINE MODELS  (line completion)  -- sorted by composite autocomplete score
  codet5_small_mlc          syntax=0.082  brackets=0.860  construct=0.800  first_tok=0.016  p50= 173.6ms
  NO-LOAD-codet5_small      syntax=0.084  brackets=0.852  construct=0.800  first_tok=0.000  p50= 800.9ms
  codet5_small              syntax=0.056  brackets=0.806  construct=0.850  first_tok=0.014  p50= 311.4ms
  NO-LOAD-codet5_small_mlc  syntax=0.072  brackets=0.796  construct=0.800  first_tok=0.00

## Optional - save full JSON aggregate of all results

In [15]:
import json
os.makedirs(f'{EVAL_ROOT}/comparisons', exist_ok=True)
aggregate = {
    'token_models': token_results,
    'line_models':  line_results,
}
with open(f'{EVAL_ROOT}/comparisons/all_results.json', 'w') as f:
    json.dump(aggregate, f, indent=2, default=str)
print(f'Aggregate saved -> {EVAL_ROOT}/comparisons/all_results.json')

Aggregate saved -> C:\Users\Roman\Documents\Projects\code_autocomplete/results/comparisons/all_results.json
